# Transparent GPU-Hour Allocation

This notebook runs Team A's exact six-team comparison: random-order FCFS versus a carbon-aware VCG priority mechanism.

For each team, estimated emissions are $e_i = 0.24d_i$ and the selection score is $v_i - 0.5e_i$. The mechanism chooses the feasible group with the highest total score, subject to 100 GPU-hours.


In [ ]:
# Make the solver available both locally and when this notebook is opened in Google Colab.
from pathlib import Path
import os
import subprocess
import sys

repository_url = 'https://github.com/GihoonE/COMPSCI206-PS2.git'
repository_name = 'COMPSCI206-PS2'
project_root = Path.cwd()

if not (project_root / 'src' / 'gpu_allocation.py').exists():
    clone_path = project_root / repository_name
    if not clone_path.exists():
        subprocess.run(['git', 'clone', repository_url, repository_name], check=True)
    os.chdir(clone_path)
    project_root = Path.cwd()

assert (project_root / 'src' / 'gpu_allocation.py').exists()
sys.path.insert(0, str(project_root))

from src.gpu_allocation import (
    CAPACITY_GPU_HOURS,
    default_example,
    fcfs_allocation,
    outcome_metrics,
    team_rows,
    vcg_allocation,
)


In [ ]:
# The six teams use the project's Low / Medium / High values: 3, 6, and 9.
teams, arrival_order = default_example()
fcfs = fcfs_allocation(teams, arrival_order, CAPACITY_GPU_HOURS)
vcg = vcg_allocation(teams, CAPACITY_GPU_HOURS)

fcfs_summary = outcome_metrics(teams, fcfs)
vcg_summary = outcome_metrics(teams, vcg)

print('FCFS arrival order:', ' -> '.join(teams[i].name for i in arrival_order))
print('FCFS selected teams:', fcfs_summary['selected_teams'])
print('VCG selected teams:', vcg_summary['selected_teams'])


In [ ]:
# Compact comparison used in the paper, poster, and Hugging Face audit.
for label, summary in [('FCFS', fcfs_summary), ('Carbon-aware VCG', vcg_summary)]:
    print(f'\n{label}')
    for key in ['teams_served', 'gpu_hours_used', 'unused_gpu_hours',
                'total_true_project_value', 'carbon_adjusted_true_score',
                'total_estimated_emissions_kg_co2e',
                'total_priority_payment_credits']:
        print(f'  {key}: {summary[key]}')


In [ ]:
# Transparent allocation audit: one row for every team under the VCG rule.
for row in team_rows(teams, vcg):
    print(row)

# VCG priority credits are the externality each selected team creates for others.
# The separate run_simulation.py file writes these same results to CSV files.
